# ScreamingFace · DRACO Lite

Run two real DRACO research questions through one OpenRouter-backed Fusion, then grade the Fusion
and both members with DRACO's official per-criterion judge prompt.

This uses **`draco-lite@1`**: the first two cases from the pinned DRACO dataset, every rubric
criterion for those cases, and two independent judge passes per criterion. It exercises the real
research, synthesis, grading, and aggregation protocol, but its two-case result is **not comparable
to a production DRACO score**. Production `draco@1` uses all 100 cases and five judge passes.

## Before you run it

```bash
cd packages/screamingface/apps/screamingface-engine
export HF_TOKEN=hf_...  # accepted DRACO dataset access
./dev.sh restart
```

Connect one OpenRouter API key below. The SDK speaks only to the ScreamingFace engine; the engine
calls AI Gateway for inference and uses OpenRouter's managed search/fetch tools.

## 1 · Connect

In [1]:
import screamingface as sf

sf.connect()

PanelWidget(children=(HTML(value='<style>\n.sf-ui {\n  --sf-bg:#ffffff;--sf-surface:#f6f6f7;--sf-surface-2:#ef…

Connect **OpenRouter**. The same engine-scoped connection covers all three model
routes below. Dataset access is separate: `HF_TOKEN` belongs in the engine environment.

## 2 · Compose

In [2]:
ANSWER_PROMPT = "Answer thoroughly. Use web evidence and cite sources."
SYNTHESIS_PROMPT = "Combine the panel answers into one stronger answer."

gpt = sf.Model(
    "openrouter/openai/gpt-5.5",
    name="gpt",
    prompt=ANSWER_PROMPT,
    params={"temperature": 0, "max_tokens": 4096},
)
opus = sf.Model(
    "openrouter/anthropic/claude-opus-4.8",
    name="opus",
    prompt=ANSWER_PROMPT,
    params={"temperature": 0, "max_tokens": 4096},
)

fusion = sf.Fusion(
    "research-duo",
    members=[gpt, opus],
    reducer=sf.reducers.Model(
        model="openrouter/openai/gpt-5.5",
        prompt=SYNTHESIS_PROMPT,
        params={"temperature": 0, "max_tokens": 4096},
    ),
)

fusion

Fusion(name='research-duo', members=(Model(name='gpt', model='openrouter/openai/gpt-5.5', prompt='Answer thoroughly. Use web evidence and cite sources.'), Model(name='opus', model='openrouter/anthropic/claude-opus-4.8', prompt='Answer thoroughly. Use web evidence and cite sources.')), reducer=Model(model='openrouter/openai/gpt-5.5', prompt='Combine the panel answers into one stronger answer.'))

Construction is local and makes no model calls. The benchmark—not the Fusion—adds
`web_search`, `web_fetch`, the twelve-call budget, leak-domain exclusions, grader, and aggregator.

## 3 · Evaluate two real cases

**Spend warning:** a typical DRACO case has roughly forty criteria. Two cases × two passes × the
Fusion plus two distinct member answers is roughly **480 judge requests**, in addition to the
researched answer and synthesis calls. This is intentionally smaller than production, not free.

In [3]:
draco = sf.benchmarks.load("draco-lite@1")
report = draco.evaluate(fusion)

HTML(value='<style>\n.sf-ui {\n  --sf-bg:#ffffff;--sf-surface:#f6f6f7;--sf-surface-2:#efeff1;\n  --sf-ink:#161…

EngineProtocolError: URL4 engine returned HTTP 502 (benchmark_evaluation_failed): ResolutionError: AI Gateway returned HTTP 500 (provider_unavailable) for 'openrouter/openai/gpt-5.5'

One SDK call sends one complete URL4 expression to `GET /v1?q=...`. For this
two-member Fusion, each case performs two researched member answers, one synthesis, then two
independent judge passes for every rubric criterion across the Fusion and both member answers.

## 4 · Compare

In [ ]:
report

`score`, `baseline`, and `gain` use the same paired case and rubric verdicts.
Inspect the exact transaction with `report.url4`. Switching to `draco@1` changes the benchmark
itself to all 100 cases and five judge passes; the Fusion construction does not change.